# **LangGraph — Introduction to Graph Structure**

## Outline
* What is LangGraph and why do we need it?
* Three core concepts: **State** · **Node** · **Edge**
* Build the first simple graph (without an LLM)
* Conditional Edge — Conditional routing
* Loop in the graph
* Checkpointing with InMemorySaver

## 0. Installation

```bash
pip install langgraph
```

## 1. What is LangGraph?

**LangGraph** is a framework for building AI applications with **complex control flow**.

```
LangChain (simple chain):
  Input → Step1 → Step2 → Step3 → Output
  (linear, without conditions, without memory)

LangGraph (graph):
  Input → Node A
              ├─[condition 1]→ Node B → Node D
              └─[condition 2]→ Node C ↩ (loop)
  (conditional, with loops, with persistent memory)
```

### Why LangGraph?

| Need | Appropriate tool |
|------|-------------|
| A simple LLM call | `init_chat_model` |
| A linear sequence of steps | LCEL chain |
| Agent with tool-calling | `create_agent` |
| **Complex control flow, loops, human approval** | **LangGraph** |

## 2. Three Core Concepts

```
┌─────────────────────────────────────────────┐
│                  STATE                       │
│   {"messages": [...], "counter": 3, ...}    │
│   (shared dictionary between all Nodes)           │
└─────────────────────────────────────────────┘
          ↑ read/write
          │
   ┌──────┴───────┐
   │    NODE A    │  ← Python function that receives State
   │ (function)  │     and updates part of the State
   └──────┬───────┘
          │
        EDGE  ← connection between Nodes
          │    (direct or conditional)
   ┌──────┴───────┐
   │    NODE B    │
   └──────────────┘
```

- **State**: shared dictionary — every Node can read and write it
- **Node**: Python function — receives `state` and returns an update dictionary
- **Edge**: connection — direct (`add_edge`) or conditional (`add_conditional_edges`)

## 3. First Graph — Linear (without an LLM)

In [ ]:
# Example for TypeDict
from typing import TypedDict

# Define a TypedDict for a person
class Person(TypedDict):
    name: str
    age: int
    city: str

# Create a valid Person object
person1: Person = {
    "name": "Alice",
    "age": 30,
    "city": "New York"
}

# Access fields like a regular dictionary
print(f"Name: {person1['name']}")
print(f"Age: {person1['age']}")
print(f"City: {person1['city']}")

# Iterate through the TypedDict
for key, value in person1.items():
    print(f"{key}: {value}")

Name: Alice
Age: 30
City: New York
name: Alice
age: 30
city: New York


In [10]:
from typing import TypedDict, Annotated, Literal

# Define a TypedDict with Annotated and Literal
class User(TypedDict):
    # Literal restricts to specific string values
    role: Literal["admin", "user", "guest"]
    
    # Annotated adds metadata/constraints (here just documentation)
    age: Annotated[int, "Age in years, must be between 0 and 150"]
    
    # Literal with multiple options
    status: Literal["active", "inactive", "suspended"]
    
    # Annotated with Literal for validation hints
    gender: Annotated[
        Literal["male", "female", "non-binary", "prefer-not-to-say"],
        "Gender identity options"
    ]

# Valid user
user1: User = {
    "role": "admin",
    "age": 30,
    "status": "active",
    "gender": "female"
}

print("✅ Valid User:")
print(f"  Role: {user1['role']}")
print(f"  Age: {user1['age']}")
print(f"  Status: {user1['status']}")
print(f"  Gender: {user1['gender']}")

# This would cause a type error (uncomment to see)
# invalid_user: User = {
#     "role": "superuser",  # ❌ Not in Literal
#     "age": 30,
#     "status": "active",
#     "gender": "female"
# }

✅ Valid User:
  Role: admin
  Age: 30
  Status: active
  Gender: female


In [11]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

# ============================================
# 1. DEFINE THE MODEL WITH PYDANTIC
# ============================================

class User(BaseModel):
    # Literal restricts to specific string values
    role: Literal["admin", "user", "guest"]
    
    # Field adds validation constraints!
    age: int = Field(
        ge=0,                    # Greater than or equal to 0
        le=150,                  # Less than or equal to 150
        description="Age in years, must be between 0 and 150"
    )
    
    # Literal with multiple options
    status: Literal["active", "inactive", "suspended"]
    
    # Literal with validation
    gender: Literal["male", "female", "non-binary", "prefer-not-to-say"]

# ============================================
# 2. CREATE VALID USERS
# ============================================

print("=" * 50)
print("✅ CREATING VALID USERS")
print("=" * 50)

# Valid user
try:
    user1 = User(
        role="admin",
        age=30,
        status="active",
        gender="female"
    )
    print(f"✅ User created successfully:")
    print(f"  Role: {user1.role}")
    print(f"  Age: {user1.age}")
    print(f"  Status: {user1.status}")
    print(f"  Gender: {user1.gender}")
except ValidationError as e:
    print(f"❌ Validation Error: {e}")


✅ CREATING VALID USERS
✅ User created successfully:
  Role: admin
  Age: 30
  Status: active
  Gender: female


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ── 1. Define State ──────────────────────────────────────
class MyState(TypedDict):
    number: int
    result: str

# ── 2. Define Nodes ─────────────────────────────────────
def double_it(state: MyState) -> dict:
    """First Node: doubles the number"""
    new_val = state["number"] * 2
    print(f"  [double_it]  {state['number']} × 2 = {new_val}")
    return {"number": new_val}

def describe_it(state: MyState) -> dict:
    """Second Node: writes a description"""
    text = f"Final number: {state['number']}"
    print(f"  [describe_it]  {text}")
    return {"result": text}

# ── 3. Build the graph ────────────────────────────────────────
graph_builder = StateGraph(MyState)

# Add Nodes
graph_builder.add_node("double", double_it)
graph_builder.add_node("describe", describe_it)

# Add Edges (execution flow)
graph_builder.add_edge(START, "double")    # Start here
graph_builder.add_edge("double", "describe")
graph_builder.add_edge("describe", END)    # End here

# ── 4. Compile ──────────────────────────────────────────
graph = graph_builder.compile()

# ── 5. Run ─────────────────────────────────────────────
print("=== Graph Execution ===")
result = graph.invoke({"number": 5, "result": ""})
print(f"\nFinal output: {result}")

=== Graph Execution ===
  [double_it]  5 × 2 = 10
  [describe_it]  Final number: 10

Final output: {'number': 10, 'result': 'Final number: 10'}


### Important Notes:

```
Graph execution:
  START → [double_it] → [describe_it] → END

State at each step:
  Input:     {number: 5,  result: ""}
  After double: {number: 10, result: ""}   ← only `number` was updated
  After describe:{number: 10, result: "Final number: 10"}  ← only `result` was updated
```

> **Important**: A Node returns only the fields that changed, not the entire State.

## 4. Conditional Edge — Conditional Routing

After a Node, we decide **based on the State** where to go next.

In [6]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class NumberState(TypedDict):
    number: int
    label: str

# ── Nodes ───────────────────────────────────────────────
def check_number(state: NumberState) -> dict:
    """First Node: checks the number"""
    print(f"  [check_number]  number: {state['number']}")
    return {}  # We only perform routing; the State does not change

def handle_positive(state: NumberState) -> dict:
    return {"label": f"{state['number']} is positive ✓"}

def handle_negative(state: NumberState) -> dict:
    return {"label": f"{state['number']} is negative ✗"}

def handle_zero(state: NumberState) -> dict:
    return {"label": "It is zero"}

# ── Router function ──────────────────────────────────────
# This function receives State and returns the name of the next Node
def route_number(state: NumberState) -> Literal["positive", "negative", "zero"]:
    if state["number"] > 0:
        return "positive"
    elif state["number"] < 0:
        return "negative"
    else:
        return "zero"

# ── Build the graph ───────────────────────────────────────────
builder = StateGraph(NumberState)

builder.add_node("check", check_number)
builder.add_node("positive", handle_positive)
builder.add_node("negative", handle_negative)
builder.add_node("zero", handle_zero)

builder.add_edge(START, "check")

# Conditional Edge: after `check`, the `route_number` function is called
builder.add_conditional_edges(
    "check",          # From which Node
    route_number,     # Router function
    {
        "positive": "positive",  # If it returns "positive" → Node positive
        "negative": "negative",
        "zero":     "zero",
    }
)

builder.add_edge("positive", END)
builder.add_edge("negative", END)
builder.add_edge("zero", END)

graph = builder.compile()

# ── Test ─────────────────────────────────────────────────
for n in [7, -3, 0]:
    result = graph.invoke({"number": n, "label": ""})
    print(f"  Result: {result['label']}\n")

  [check_number]  number: 7
  Result: 7 is positive ✓

  [check_number]  number: -3
  Result: -3 is negative ✗

  [check_number]  number: 0
  Result: It is zero



```
Graph structure:

  START → [check]
               ├─ number > 0 → [positive] → END
               ├─ number < 0 → [negative] → END
               └─ number = 0 → [zero]     → END
```

## 5. Reducer — When a Field Must Be Aggregated

By default, an update **replaces** the existing value.  
With `Annotated + operator.add`, we can **append/add** values.

In [12]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class LogState(TypedDict):
    value: int
    # Annotated + operator.add → each update is added (not replaced)
    log: Annotated[list[str], operator.add]

def step_a(state: LogState) -> dict:
    new_val = state["value"] + 10
    return {
        "value": new_val,
        "log": [f"step_a: {state['value']} → {new_val}"]
    }

def step_b(state: LogState) -> dict:
    new_val = state["value"] * 2
    return {
        "value": new_val,
        "log": [f"step_b: {state['value']} → {new_val}"]
    }

builder = StateGraph(LogState)
builder.add_node("a", step_a)
builder.add_node("b", step_b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

graph = builder.compile()
result = graph.invoke({"value": 5, "log": ["initial value"]})

print(f"Final value: {result['value']}")
print("Step log:")
for entry in result["log"]:
    print(f"  {entry}")

Final value: 30
Step log:
  initial value
  step_a: 5 → 15
  step_b: 15 → 30


## 6. Loop — Loop in the Graph

The graph can return to a previous Node ← this is one of LangGraph's main strengths.

In [13]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class CounterState(TypedDict):
    count: int
    max_count: int

def increment(state: CounterState) -> dict:
    new_count = state["count"] + 1
    print(f"  [increment]  count: {state['count']} → {new_count}")
    return {"count": new_count}

def should_continue(state: CounterState) -> Literal["increment", "__end__"]:
    """Router: continue or finish?"""
    if state["count"] < state["max_count"]:
        return "increment"  # ← return to the previous Node = loop
    else:
        return "__end__"    # ← END

builder = StateGraph(CounterState)
builder.add_node("increment", increment)

builder.add_edge(START, "increment")

# Conditional Edge that can return to the same Node
builder.add_conditional_edges(
    "increment",
    should_continue,
    {
        "increment": "increment",  # ← loop!
        "__end__": END
    }
)

graph = builder.compile()

print("=== Counting to 4 ===")
result = graph.invoke({"count": 0, "max_count": 4})
print(f"\nFinal count: {result['count']}")

=== Counting to 4 ===
  [increment]  count: 0 → 1
  [increment]  count: 1 → 2
  [increment]  count: 2 → 3
  [increment]  count: 3 → 4

Final count: 4


```
Graph structure with a loop:

  START → [increment] ←─────────┐
               │                │
               ├─ count < max ──┘  (loop)
               └─ count ≥ max → END
```

> This pattern is exactly the ReAct loop used by agents:
> think → call a tool → inspect the result → think again → ...

## 7. Checkpointing — Saving State

With `InMemorySaver`, a conversation can resume from where it stopped.

In [14]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class ChatState(TypedDict):
    history: Annotated[list[str], operator.add]

def add_message(state: ChatState) -> dict:
    return {}  # Display only

builder = StateGraph(ChatState)
builder.add_node("chat", add_message)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# ← Pass the checkpointer to `compile`
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)    # state persists across multiple invocations

# ── thread_id = conversation identifier ────────────────────────────
config = {"configurable": {"thread_id": "user_ali"}}

# First message
graph.invoke({"history": ["Hello! My name is Meisam."]}, config)

# Second message — the previous State is loaded automatically
graph.invoke({"history": ["How are you?"]}, config)

# View the complete State of this thread
state = graph.get_state(config)
print("Conversation history:")
for msg in state.values["history"]:
    print(f"  {msg}")

Conversation history:
  Hello! My name is Meisam.
  How are you?


## Single user chatbot with memory

In [15]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

# ============================================
# 1. INITIALIZE OLLAMA MODEL
# ============================================

# Initialize the Ollama model
model = ChatOllama(
    model="qwen3.8:latest",  
    temperature=0.7,          
)

# ============================================
# 2. DEFINE STATE
# ============================================

class ChatState(TypedDict):
    # Messages will be appended using operator.add
    messages: Annotated[list, operator.add]
    # Track how many times the LLM was called
    llm_calls: int

# ============================================
# 3. DEFINE THE CHAT NODE
# ============================================

def chat_node(state: ChatState) -> dict:
    """
    Process the conversation and generate a response using Ollama
    """
    # Get the last message (user's input)
    last_message = state["messages"][-1]
    
    # Add a system message to guide the AI
    system_message = SystemMessage(
        content="You are a helpful, friendly AI assistant. "
                "Engage in natural conversation and provide useful responses."
    )
    
    # Prepare messages for the LLM
    # Include system message + all conversation history
    messages_to_send = [system_message] + state["messages"]
    
    # Call Ollama
    print(f"🤖 Thinking...")
    response = model.invoke(messages_to_send)
    
    # Return updates to the state
    return {
        "messages": [response],  # Appended to existing messages
        "llm_calls": state.get("llm_calls", 0) + 1
    }

# ============================================
# 4. BUILD THE GRAPH WITH CHECKPOINTING
# ============================================

# Create graph builder
builder = StateGraph(ChatState)

# Add the chat node
builder.add_node("chat", chat_node)

# Define the flow: START → chat → END
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Create checkpointer for memory
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# ============================================
# 5. INTERACTIVE CHAT FUNCTION
# ============================================

def run_chatbot():
    """
    Run an interactive chatbot with memory
    """
    print("\n" + "=" * 50)
    print("🤖 Qwen Chatbot with Memory")
    print("=" * 50)
    print("Commands:")
    print("  • Type 'exit' or 'quit' to end the chat")
    print("  • Type 'history' to see the conversation history")
    print("  • Type 'stats' to see usage statistics")
    print("=" * 50 + "\n")
    
    # Create a unique thread ID for this session
    import uuid
    thread_id = f"chat_{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}
    
    # Initialize state
    state = {
        "messages": [],
        "llm_calls": 0
    }
    
    print(f"🆔 Session ID: {thread_id}")
    print("👋 Say hello!\n")
    
    while True:
        # Get user input
        user_input = input("👤 You: ").strip()
        
        # Check for exit commands
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("👋 Goodbye! Have a great day!")
            break
        
        # Check for special commands
        if user_input.lower() == "history":
            # Get and display the full conversation history
            current_state = graph.get_state(config)
            messages = current_state.values.get("messages", [])
            
            print("\n📋 Conversation History:")
            print("-" * 40)
            if not messages:
                print("  No messages yet.")
            else:
                for msg in messages:
                    if isinstance(msg, HumanMessage):
                        print(f"  👤 You: {msg.content}")
                    elif isinstance(msg, AIMessage):
                        print(f"  🤖 Assistant: {msg.content}")
            print("-" * 40 + "\n")
            continue
        
        if user_input.lower() == "stats":
            # Display usage statistics
            current_state = graph.get_state(config)
            messages = current_state.values.get("messages", [])
            llm_calls = current_state.values.get("llm_calls", 0)
            
            print("\n📊 Session Statistics:")
            print("-" * 40)
            print(f"  Messages: {len(messages)}")
            print(f"  LLM Calls: {llm_calls}")
            print(f"  Session ID: {thread_id}")
            print("-" * 40 + "\n")
            continue
        
        # Add user message to state
        state["messages"] = [HumanMessage(content=user_input)]
        
        # Invoke the graph
        try:
            result = graph.invoke(state, config)
            
            # Extract the last message (AI's response)
            last_message = result["messages"][-1]
            
            # Update state for next iteration
            state = result
            
            # Print the AI's response
            if isinstance(last_message, AIMessage):
                print(f"🤖 Assistant: {last_message.content}\n")
            else:
                print(f"🤖 Assistant: {last_message.content}\n")
                
        except Exception as e:
            print(f"❌ Error: {e}")
            print("Please try again.\n")

# ============================================
# 6. RUN THE CHATBOT
# ============================================

if __name__ == "__main__":
    run_chatbot()


🤖 Qwen Chatbot with Memory
Commands:
  • Type 'exit' or 'quit' to end the chat
  • Type 'history' to see the conversation history
  • Type 'stats' to see usage statistics

🆔 Session ID: chat_137947e1
👋 Say hello!

🤖 Thinking...
🤖 Assistant: Hi Meisam! Nice to meet you. I'm Qwen, an AI assistant. I'm here to help with whatever you need — answering questions, brainstorming ideas, writing, problem-solving, or just having a chat. How can I help you today?

🤖 Thinking...
🤖 Assistant: Thanks for letting me know, Meisam — that helps me calibrate the tone. Here's a concise overview:

## Supersymmetry (SUSY)

**The core idea:**
Supersymmetry is a proposed symmetry of nature that pairs every known particle with a heavier "superpartner." For example:

- Quarks ↔ Squarks
- Electrons ↔ Selectrons (sleptons)
- Photons ↔ Photinos
- Gluons ↔ Gluinos

Fermions (matter particles) get bosonic partners, and bosons (force carriers) get fermionic partners.

**Why physicists proposed it:**
- **Unifies force

## Multiple user chatbot with memory

In [16]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama
import uuid
import json
from datetime import datetime

# Initialize Ollama
model = ChatOllama(
    model="qwen3.8:latest",
    temperature=0.7,
)

# ============================================
# 1. EXTENDED STATE WITH MORE METADATA
# ============================================

class ChatState(TypedDict):
    messages: Annotated[list, operator.add]
    llm_calls: int
    user_id: str
    session_start: str
    total_tokens: int

# ============================================
# 2. CHAT NODE WITH SYSTEM PROMPT
# ============================================

def chat_node(state: ChatState) -> dict:
    """Process conversation with Ollama"""
    
    # Get last message
    last_message = state["messages"][-1]
    
    # Enhanced system prompt with context
    system_message = SystemMessage(
        content=f"""You are a helpful AI assistant having a conversation with a user.
        
Current conversation context:
- User ID: {state.get('user_id', 'unknown')}
- Session started: {state.get('session_start', 'unknown')}
- Number of messages: {len(state.get('messages', []))}

Guidelines:
1. Be friendly and conversational
2. Provide helpful, accurate information
3. Remember context from previous messages
4. Ask follow-up questions when appropriate
5. Keep responses concise but informative
"""
    )
    
    # Prepare messages
    messages_to_send = [system_message] + state["messages"]
    
    # Call LLM
    print(f"🤖 Thinking... (Message {len(state.get('messages', [])) + 1})")
    response = model.invoke(messages_to_send)
    
    # Count approximate tokens (rough estimate)
    total_chars = sum(len(msg.content) for msg in messages_to_send)
    approx_tokens = total_chars // 4  # Rough approximation
    
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1,
        "total_tokens": state.get("total_tokens", 0) + approx_tokens
    }

# ============================================
# 3. BUILD GRAPH
# ============================================

builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# ============================================
# 4. COMPLETE CHATBOT MANAGER
# ============================================

class ChatbotManager:
    """Manages multiple chat sessions"""
    
    def __init__(self, graph, memory):
        self.graph = graph
        self.memory = memory
        self.sessions = {}
    
    def create_session(self, user_id: str = None):
        """Create a new chat session"""
        if user_id is None:
            user_id = f"user_{uuid.uuid4().hex[:8]}"
        
        thread_id = f"session_{uuid.uuid4().hex[:12]}"
        config = {"configurable": {"thread_id": thread_id}}
        
        # Initialize session state
        initial_state = {
            "messages": [],
            "llm_calls": 0,
            "user_id": user_id,
            "session_start": datetime.now().isoformat(),
            "total_tokens": 0
        }
        
        self.sessions[thread_id] = {
            "config": config,
            "user_id": user_id,
            "state": initial_state,
            "created_at": datetime.now()
        }
        
        return thread_id, config
    
    def send_message(self, thread_id: str, message: str) -> str:
        """Send a message to a specific session"""
        if thread_id not in self.sessions:
            return "❌ Session not found. Please create a new session."
        
        session = self.sessions[thread_id]
        config = session["config"]
        state = session["state"]
        
        # Add user message
        state["messages"] = [HumanMessage(content=message)]
        
        # Invoke graph
        result = self.graph.invoke(state, config)
        session["state"] = result
        
        # Get AI response
        last_message = result["messages"][-1]
        return last_message.content
    
    def get_history(self, thread_id: str) -> list:
        """Get conversation history for a session"""
        if thread_id not in self.sessions:
            return []
        
        session = self.sessions[thread_id]
        state = self.graph.get_state(session["config"])
        return state.values.get("messages", [])
    
    def get_stats(self, thread_id: str) -> dict:
        """Get session statistics"""
        if thread_id not in self.sessions:
            return {}
        
        session = self.sessions[thread_id]
        state = session["state"]
        messages = self.get_history(thread_id)
        
        return {
            "session_id": thread_id,
            "user_id": session["user_id"],
            "messages_count": len(messages),
            "llm_calls": state.get("llm_calls", 0),
            "tokens_used": state.get("total_tokens", 0),
            "created_at": session["created_at"].isoformat(),
            "duration": str(datetime.now() - session["created_at"]).split('.')[0]
        }
    
    def list_sessions(self) -> list:
        """List all active sessions"""
        return [
            {
                "thread_id": tid,
                "user_id": session["user_id"],
                "created_at": session["created_at"].isoformat(),
                "messages": len(self.get_history(tid))
            }
            for tid, session in self.sessions.items()
        ]

# ============================================
# 5. INTERACTIVE MENU SYSTEM
# ============================================

def main():
    """Main chatbot application with menu"""
    
    manager = ChatbotManager(graph, memory)
    
    print("\n" + "=" * 60)
    print("🤖 Qwen Chatbot System")
    print("=" * 60)
    
    # Create a new session
    thread_id, config = manager.create_session()
    print(f"✅ Session created!")
    print(f"   Session ID: {thread_id}")
    
    while True:
        print("\n" + "=" * 60)
        print("📋 Menu:")
        print("  1. 💬 Chat")
        print("  2. 📋 View History")
        print("  3. 📊 View Statistics")
        print("  4. 🔄 New Session")
        print("  5. 📂 List Sessions")
        print("  6. 👋 Exit")
        print("=" * 60)
        
        choice = input("Select option (1-6): ").strip()
        
        if choice == "1":  # Chat
            print("\n💬 Chat Mode (type 'back' to return to menu)")
            print("-" * 40)
            
            while True:
                user_input = input("👤 You: ").strip()
                
                if user_input.lower() in ["back", "menu"]:
                    break
                
                if not user_input:
                    continue
                
                # Send message
                response = manager.send_message(thread_id, user_input)
                print(f"🤖 Assistant: {response}")
                print()
        
        elif choice == "2":  # View History
            print("\n📋 Conversation History:")
            print("-" * 40)
            
            messages = manager.get_history(thread_id)
            if not messages:
                print("  No messages yet.")
            else:
                for msg in messages:
                    if isinstance(msg, HumanMessage):
                        print(f"  👤 You: {msg.content}")
                    elif isinstance(msg, AIMessage):
                        print(f"  🤖 Assistant: {msg.content}")
            print("-" * 40)
        
        elif choice == "3":  # Statistics
            stats = manager.get_stats(thread_id)
            print("\n📊 Session Statistics:")
            print("-" * 40)
            for key, value in stats.items():
                print(f"  {key.replace('_', ' ').title()}: {value}")
            print("-" * 40)
        
        elif choice == "4":  # New Session
            thread_id, config = manager.create_session()
            print("\n✅ New session created!")
            print(f"   Session ID: {thread_id}")
        
        elif choice == "5":  # List Sessions
            sessions = manager.list_sessions()
            print("\n📂 Active Sessions:")
            print("-" * 40)
            if not sessions:
                print("  No active sessions.")
            else:
                for s in sessions:
                    print(f"  • {s['thread_id']}")
                    print(f"    User: {s['user_id']}")
                    print(f"    Messages: {s['messages']}")
                    print(f"    Created: {s['created_at']}")
            print("-" * 40)
        
        elif choice == "6":  # Exit
            print("\n👋 Goodbye! Have a great day!")
            break
        
        else:
            print("❌ Invalid option. Please try again.")

# ============================================
# 6. RUN THE APPLICATION
# ============================================

if __name__ == "__main__":
    main()


🤖 Qwen Chatbot System
✅ Session created!
   Session ID: session_5321b856782c

📋 Menu:
  1. 💬 Chat
  2. 📋 View History
  3. 📊 View Statistics
  4. 🔄 New Session
  5. 📂 List Sessions
  6. 👋 Exit

💬 Chat Mode (type 'back' to return to menu)
----------------------------------------
🤖 Thinking... (Message 2)
🤖 Assistant: Hi Meisam! Nice to meet you! 😊

I'm an AI assistant here to help you with all sorts of things — whether that's answering questions, brainstorming ideas, writing, coding, translating, or just having a good conversation. No topic is off-limits, and I'll do my best to give you accurate, useful answers.

So, what's on your mind today, Meisam? Is there something I can help you with, or did you just feel like saying hi? Either way, I'm happy to chat!

🤖 Thinking... (Message 4)
🤖 Assistant: Agent AI refers to AI systems that can autonomously perceive their environment, plan, and take multi-step actions to achieve specific goals — often using tools like APIs, code execution, or we

## Summary

```
LangGraph = StateGraph + Nodes + Edges

State      → Shared TypedDict between all Nodes
Node       → Python function: receives state and returns an update dict
Edge       → add_edge("a", "b")  [direct]
Cond. Edge → add_conditional_edges("a", router_fn, {...})  [conditional]
Reducer    → Annotated[list, operator.add]  for aggregating fields
Loop       → Conditional Edge that returns to a previous Node
Checkpoint → compile(checkpointer=InMemorySaver())  for persistent memory
```

In the next notebook, we will combine this structure with an **LLM** and build a real chatbot.